<a href="https://colab.research.google.com/github/evinracher/3010090-ontological-engineering/blob/main/week5/workshop/part2/Taller_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Taller: Multi-Modal RAG con CLIP + Gemini (PDF con texto e imágenes)

Estructura (igual al taller de referencia):
- **Ejemplo** (código completo)
- **✍️ Tu turno** (espacio para completar)
- **❓ Preguntas** (qué pasaría si…)

**Objetivo:** Construir un mini Multi-Modal RAG que:
1) Extrae **texto e imágenes** desde un PDF  
2) Genera embeddings con **CLIP** (texto ↔ imagen en el mismo espacio)  
3) Indexa en **FAISS** con metadata (modalidad, página, etc.)  
4) Hace retrieval híbrido (texto+imagen) y responde con **Gemini** usando evidencia

## 1. Setup

In [20]:
!pip -q install -U langchain_google_genai python-dotenv pymupdf tools langchain_community langchain langchain_core faiss-cpu

In [21]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI


In [22]:
from dotenv import load_dotenv
load_dotenv()

# Extraer API KEY
try:
    from google.colab import userdata
    _k = userdata.get('GOOGLE_API_KEY')
    if _k:
        os.environ['GOOGLE_API_KEY'] = _k
except Exception:
    pass

# Inicializar el modelo Clip para embeddings unificados
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [23]:
# Inicializar LLM

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.2
)

print("modelo listo")

modelo listo


## 2. Crear funciones para generar embeddings usando CLIP

In [24]:
def embed_text(text):
    """Crear embedding de texto usando CLIP"""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77  # CLIP max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        # Normalizar embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()

def embed_image(image_data):
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data.convert("RGB")

    inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        features = outputs.pooler_output
        features = features / features.norm(dim=-1, keepdim=True)

    return features.squeeze().cpu().numpy()

### ✍️ Tu turno

Prueba `embed_image` con una imagen artificial (sin depender del PDF).


In [25]:
# TODO: crea una imagen simple con PIL (ej: un cuadrado blanco) y genera su embedding *
from PIL import Image

img = Image.new("RGB", (224, 224), color="white")
embedding = embed_image(img)

print(embedding.shape)
print(embedding[:10])

(768,)
[ 0.01771418  0.01468623 -0.00405743  0.01474181  0.00937039 -0.05014379
  0.04683543  0.0063223   0.00083838  0.01578645]


### ❓ Preguntas
- ¿Qué pasa si pasas una imagen en escala de grises (modo "L")? ¿por qué el código hace `.convert("RGB")` en otros lugares?

R/= El modelo puede fallar ya que fue entrenado en RGB

- ¿Qué significa que CLIP produzca un vector en ℝ^d? (intuitivo: “coordenadas” semánticas)

R/= d son las dimensiones semánticas en las que la imagen va a estar representada. Las imagénes similares coordenadas en esas dimensiones serán cercanas o estarán relacionadas semánticamente.

- ¿Por qué embeddings visuales y textuales pueden compararse con coseno si viven en el mismo espacio?

R/= Por que CLIP representa imágenes y textos en un mismo espacio vectorial y se puede usar el coseno para comparar que tan relacionados están sus representaciones.


## 3. Lectura del PDF a procesar

In [26]:
# Procesar PDF
pdf_path="/content/1706.03762v7.pdf"
doc=fitz.open(pdf_path)

# Guardar todos los documentos y embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Guardar datos de la imagen para el LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

### ✍️ Tu turno

Explora el PDF: páginas, texto e imágenes.


In [27]:
# TODO: imprime número de páginas
print(len(doc))


# TODO: imprime un snippet del texto de la primera página
first_page = doc[0]
text = first_page.get_text()
print(text[:300])


# TODO: cuenta imágenes por página (dict page->count) usando page.get_images(full=True)
images_per_page = {i: len(doc[i].get_images(full=True)) for i in range(len(doc))}
print(images_per_page)


15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par
{0: 0, 1: 0, 2: 1, 3: 2, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0}


### ❓ Preguntas
- ¿Qué pasa si el PDF es un escaneo? (texto vacío, pero imágenes sí)

R/= Puede que no se encuentre texto en el pdf. Se tendría que implementar OCR para poder procesar las imágenes y sacar texto de los PDFs.

- ¿Qué cambiarías para soportar PDFs muy grandes (100+ páginas) sin quedarte sin RAM?


R/= Se puede procesar por pagínas o por lotes.

- ¿En qué casos conviene chunking “layout-aware” en vez de chunking por caracteres?

R/= Cuando la estructura del PDF importa, por ejemplo cuando se tienen articulos doble columna. El layout-aware respecta la estructura del PDF y divide mejor los chunks en esos casos.

## 4. Procesar con CLIP

In [28]:
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def embed_text(text: str) -> np.ndarray:
    inputs = clip_processor(text=[text], images=None, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = clip_model.get_text_features(**inputs)

    # Si devuelve un objeto, sacamos el tensor correcto
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        feats = out.pooler_output
    elif isinstance(out, (tuple, list)):
        feats = out[0]
    else:
        feats = out  # ya era tensor

    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.squeeze(0).detach().cpu().numpy()


@torch.no_grad()
def embed_image(pil_image) -> np.ndarray:
    inputs = clip_processor(text=None, images=pil_image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    out = clip_model.get_image_features(**inputs)

    # Manejar salida tipo objeto
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        feats = out.pooler_output
    elif isinstance(out, (tuple, list)):
        feats = out[0]
    else:
        feats = out

    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.squeeze(0).detach().cpu().numpy()

### ✍️ Tu turno

Embebe 3 consultas en texto y compara similitud (coseno) entre ellas.


In [32]:
def embed_text(text):
    device = next(clip_model.parameters()).device
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clip_model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        features = outputs.pooler_output
        features = features / features.norm(dim=-1, keepdim=True)

    return features.squeeze().cpu().numpy()

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

# TODO: crea 3 queries sobre el PDF *
queries = [
    "¿De qué trata el documento?",
    "¿Cuáles son los temas principales del PDF?",
    "¿Qué imágenes o figuras aparecen en el documento?"
]

query_embeddings = [embed_text(q) for q in queries]

# TODO: matriz de similitud coseno *
similarity_matrix = cosine_similarity(query_embeddings)

print("Queries:")
for i, q in enumerate(queries):
    print(f"{i+1}. {q}")

print("\nMatriz de similitud coseno:")
print(similarity_matrix)

Queries:
1. ¿De qué trata el documento?
2. ¿Cuáles son los temas principales del PDF?
3. ¿Qué imágenes o figuras aparecen en el documento?

Matriz de similitud coseno:
[[0.99999976 0.69681543 0.911999  ]
 [0.69681543 1.0000002  0.68772703]
 [0.911999   0.68772703 1.        ]]


### ❓ Preguntas
- Si dos queries son muy parecidas, ¿esperas sim más alto? ¿por qué?

R/= el coseno mide el ángulo entre dos vectores. Los vectores más alineados tienen una similitud más alta y por tanto el valor es mayor.

- ¿Qué pasa si la query es muy larga? ¿qué hace `truncation=True`?

R/= Se tiene que cortar, ya que puede pasar el número de tokens permitidos por el usuario. truncation=True hace esto automáticamente

- ¿Por qué es común normalizar embeddings antes de comparar por coseno?

R/= Para poder comparar por dirección solamente y que la magnitud no se tenga en cuenta. Normalizar convierte los vectores a vectores unitarios.


In [34]:
for i,page in enumerate(doc):
    # Procesar texto
    text=page.get_text()
    if text.strip():
        # Crear documento temporal para splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        # Crear embedding de cada chunk usando CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    # Procesar imagenes
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convertir a imagen PIL
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Crear identificador único
            image_id = f"page_{i}_img_{img_index}"

            # Guardar imagenes en base64
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64

            # Crear embedding usando CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            # Crear documento para la imagen
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

Error processing image 0 on page 2: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same
Error processing image 0 on page 3: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same
Error processing image 1 on page 3: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same


### ✍️ Tu turno

Verifica que se generaron documentos y embeddings, y revisa metadata.


In [35]:
# TODO: imprime cuántos docs quedaron
print(len(all_docs))

# TODO: muestra 3 metadatas distintas
for doc in all_docs[:3]:
    print(doc.metadata)

103
{'page': 0, 'type': 'text'}
{'page': 0, 'type': 'text'}
{'page': 0, 'type': 'text'}


### ❓ Preguntas
- ¿Qué pasa si no guardas metadata como `type` y `page`? ¿cómo afecta filtros/explicabilidad?

R/= No podría haber trazabilidad de donde salió la respuesta. La metadata sirve para describir el origen y el tipo de los chunks.

- ¿Por qué en multimodal conviene guardar “evidencia” (ej: bytes de imagen) aparte (`image_data_store`)?


R/= Para poder mostrar la imagen en su calidad original, y separar la búsqueda de la visualización, ya que los embeddings solo sirven para hacer la búsqueda, no para mostrar las imágenes originales.

- ¿Cuál es el riesgo de chunking demasiado pequeño vs demasiado grande?

R/= Si es muy pequeño se puede perder contexto importante o cortar ideas y quedar incompletas. Si es muy grande, hay ruido y consume más tokens.


## 5. Usar FAISS para almacenar los vectores

In [36]:
# Cree un almacén de vectores FAISS unificado con embeddings CLIP
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[ 0.05983447,  0.01908169,  0.00101628, ...,  0.01755678,
        -0.01347141, -0.03474744],
       [ 0.04307959, -0.01367543,  0.02963489, ...,  0.081068  ,
        -0.02662119, -0.01634874],
       [ 0.03402079,  0.0154711 ,  0.02738747, ...,  0.04109535,
        -0.00724574, -0.05862856],
       ...,
       [ 0.02235748, -0.05747142,  0.04361735, ...,  0.03141433,
        -0.00331757, -0.02133554],
       [ 0.05887604, -0.02000537,  0.00862288, ..., -0.00699059,
         0.0001912 , -0.02378349],
       [ 0.05217335, -0.06950595,  0.03980032, ...,  0.02993823,
        -0.01636731, -0.00315566]], dtype=float32)

In [38]:
# Crear un índice FAISS personalizado ya que tenemos embeddings precalculados
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

### ✍️ Tu turno

Haz una búsqueda simple y verifica que retorna items con metadata coherente.


In [41]:
# TODO: corre una búsqueda por texto usando el vector_store
query = "¿De qué trata el documento?"
query_embedding = embed_text(query)

results = vector_store.similarity_search_by_vector(query_embedding, k=5)

# TODO: imprime tipo y página de cada resultado
for r in results:
    print(f"type: {r.metadata.get('type')}, page: {r.metadata.get('page')}")

type: text, page: 13
type: text, page: 14
type: text, page: 8
type: text, page: 1
type: text, page: 2


### ❓ Preguntas
- ¿Qué pasa si en vez de un índice unificado haces dos índices separados (texto e imagen)? ¿qué ganas y qué pierdes?
- ¿Qué síntomas ves cuando el retrieval trae “cerca” cosas irrelevantes? (y qué harías: rerank, filtros, mejor chunking)


## 6. Funciones para RAG multimodal

- Retrieve

In [42]:
def retrieve_multimodal(query, k=5):
    """Retrieval unificado usando embeddings CLIP para texto e imagenes"""
    # Crear embedding de la query usando CLIP
    query_embedding = embed_text(query)

    # Buscar en almacén unificado
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

### ✍️ Tu turno

Experimenta con `k` y observa cómo cambia el contexto recuperado.


In [43]:
# TODO: prueba retrieve_multimodal con k=2 y k=8 y compara
results_k2 = retrieve_multimodal("¿De qué trata el documento?", k=2)
results_k8 = retrieve_multimodal("¿De qué trata el documento?", k=8)

# TODO: imprime (type, page) para cada set
print("Resultados con k=2:")
for r in results_k2:
    print((r.metadata.get("type"), r.metadata.get("page")))

print("\nResultados con k=8:")
for r in results_k8:
    print((r.metadata.get("type"), r.metadata.get("page")))

Resultados con k=2:
('text', 13)
('text', 14)

Resultados con k=8:
('text', 13)
('text', 14)
('text', 8)
('text', 1)
('text', 2)
('text', 1)
('text', 5)
('text', 4)


### ❓ Preguntas
- ¿Qué pasa cuando k es muy grande? (ruido + tokens + costo)

R/= Se traen más documentos lo cual consume más tokens (incrementa el costo), más memoria y además introduce ruido innecesario.

- ¿Por qué a veces conviene “traer candidatos” con k grande y luego rerankear?

R/= Porque la búsquedad vectorial es rápida pero puede traer resultados ordenados en orden incorrecto. Por eso una vez se tengan los resultados de manera rápida, conviene reordenar los resultados.

- ¿Cómo usarías metadata filtering (solo imágenes / solo página 1) si el vectorstore lo permite?

R/= Se podría filtrar exactamente de donde se quieren traer los documentos (por ejemplo, solo información de la página 1) o si quieres traer solo imágenes. Serviría por ejemplo cuando se quiere obtener información de un capítulo específico de un libro, podrías delimitarlo por las páginas en donde está ese capítulo.


- Convertir imagen al formato aceptado por el llm

In [44]:
import io, base64
from PIL import Image

def pil_to_data_url(pil_img: Image.Image, mime: str = "image/png") -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:{mime};base64,{b64}"

- Crear el mensaje multimodal

In [45]:
def create_multimodal_message(query, retrieved_docs):
    """Crear mensaje multimodal: texto + imagenes (como image_url) para ChatGoogleGenerativeAI."""
    parts = []

    # Separar documentos de texto e imagenes
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # Construir contexto de texto
    parts.append(f"Question: {query}\n\nContext:\n")

    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata.get('page','?')}]: {doc.page_content}"
            for doc in text_docs
        ])
        parts.append(f"Text excerpts:\n{text_context}\n")

    # Recolectar imagenes como PIL
    images = []
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            try:
                img_bytes = base64.b64decode(image_data_store[image_id])
                img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                images.append(img)
                parts.append(f"\n[Image from page {doc.metadata.get('page','?')}]\n")
            except Exception as e:
                parts.append(f"\n[Image from page {doc.metadata.get('page','?')} could not be decoded: {e}]\n")

    parts.append("\n\nPlease answer the question based on the provided text and images.")
    prompt = "".join(parts)

    # Enviar imagenes como image_url (data URL)
    content = [{"type": "text", "text": prompt}]
    for img in images:
        data_url = pil_to_data_url(img, mime="image/png")
        content.append({"type": "image_url", "image_url": data_url})

    return HumanMessage(content=content)

### ✍️ Tu turno

Inspecciona el mensaje multimodal antes de enviarlo al LLM (estructura de `msg.content`).


In [46]:
# TODO: crea un mensaje con 3 docs recuperados y mira su estructura
retrieved_docs = retrieve_multimodal("¿De qué trata el documento?", k=3)
message = create_multimodal_message("¿De qué trata el documento?", retrieved_docs)

print(message)

# TODO: imprime tipos de cada parte (text / image_url) y un preview corto
for i, part in enumerate(message.content):
    part_type = part.get("type")
    print(f"Parte {i+1}: {part_type}")

    if part_type == "text":
        print(part.get("text", "")[:200])
    elif part_type == "image_url":
        print(part.get("image_url", "")[:100])

    print()

content=[{'type': 'text', 'text': 'Question: ¿De qué trata el documento?\n\nContext:\nText excerpts:\n[Page 13]: The\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n\n[Page 14]: The\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,\nin\nmy\nopinion\n.\n<EOS>\n<pad>\nThe\nLaw\nwill\nnever\nbe\nperfect\n,\nbut\nits\napplication\nshould\nbe\njust\n-\nthis\nis\nwhat\nwe\nare\nmissing\n,

### ❓ Preguntas
- ¿Qué pasa si pasas demasiadas imágenes en un solo prompt?

R/= Se genera más ruido y el modelo puede ignorar algunas de las imágenes. Además resulta más costoso procesar varias imágenes a la vez.

- ¿Por qué es importante serializar la imagen como `data:image/png;base64,...`?

R/= Para mandar una representación textual de la imagen que un modelo puede entender. Este es el estandar que usan los modelos para manejar imágenes.

- ¿Qué diferencia hay entre “poner la imagen” y “describir la imagen”?

R/= Poner la imagen es cargar la imagen para que el modelo la analice. Describir la imagen sería darle una interpretación textual de la imagen, que puede ser subjetiva y no tan precisa.


- Pipeline completo del RAG multimodal

In [47]:
def invoke_multimodal(llm, msg: HumanMessage):
    try:
        return llm.invoke([msg])
    except ValueError as e:
        s = str(e)
        # Si la versión del llm usado no acepta image_url como string, lo convertimos a {"url": ...}
        if "Unrecognized message part type" in s or "image_url" in s:
            fixed = []
            for part in msg.content:
                if isinstance(part, dict) and part.get("type") == "image_url":
                    iu = part.get("image_url")
                    if isinstance(iu, str):
                        fixed.append({"type": "image_url", "image_url": {"url": iu}})
                    else:
                        fixed.append(part)
                else:
                    fixed.append(part)
            return llm.invoke([HumanMessage(content=fixed)])
        raise

In [48]:
def multimodal_pdf_rag_pipeline(query):
    context_docs = retrieve_multimodal(query, k=5)
    msg = create_multimodal_message(query, context_docs)

    response = invoke_multimodal(llm, msg)
    return getattr(response, "content", str(response))

### ✍️ Tu turno

Crea una mini-evaluación: ejecuta 3 queries y guarda resultados en una tabla simple.


In [51]:
import pandas as pd

if __name__ == "__main__":
    queries = [
        "What is the main contribution of the Transformer proposed in the paper?",
        "How does the Transformer architecture differ from recurrent and convolutional models?",
        "What results does the paper report on machine translation tasks?"
    ]

    rows = []

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)

        rows.append({
            "query": query,
            "answer": answer
        })

    results_df = pd.DataFrame(rows)
    print("\nTabla de resultados:")
    print(results_df)


Query: What is the main contribution of the Transformer proposed in the paper?
--------------------------------------------------
Answer: [{'type': 'text', 'text': 'Based on the provided text, the main contribution of the Transformer is that it is the **first transduction model to rely entirely on self-attention** to compute representations of its input and output, completely removing the need for sequence-aligned recurrent neural networks (RNNs) or convolution.\n\nKey aspects of this contribution include:\n\n*   **Eschewing Recurrence:** By replacing recurrence with an attention mechanism, the model can draw global dependencies between input and output more effectively.\n*   **Increased Parallelization:** The architecture allows for significantly more parallelization during training compared to traditional recurrent models.\n*   **Efficiency and Performance:** It can reach a new state of the art in translation quality with significantly less training time (e.g., being trained for as 

### ❓ Preguntas
- ¿Qué pasa si el modelo “infiere” un valor de las imagenes pero no está en el contexto? ¿cómo lo detectas?

R/= Esto sería una alucinación, el modelo estaría inventando la respuesta en lugar de usar la evidencia para responder. Se puede detectar comparando los números o datos de la respuesta, con el contexto recuperado.

- ¿Qué señales indican que necesitas mejorar parsing de PDF?

R/= Texto incompleto, tablas y figuras faltantes.

- ¿Cómo harías “citas” más confiables?

R/= Se puede incluir metadata que diga de que página o figura salió cada afirmación. También incluir en el mensaje los links o referencias de donde salió la afirmación (como hace chat GPT).


- Pruebas

### ✍️ Tu turno

Modifica SOLO las queries (no el pipeline) para forzar retrieval de imágenes y luego de texto.


In [52]:
# TODO: escribe 2 queries:
# 1) una que claramente apunte a un gráfico/figura (para traer imágenes)
# 2) una que apunte a definiciones o texto (para traer chunks textuales)

query_figure = "What does Figure 1 show about the Transformer architecture?"
query_text = "What is the definition of scaled dot-product attention in the paper?"

queries = [query_figure, query_text]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 60)

    retrieved = retrieve_multimodal(query, k=5)

    print("Retrieval:")
    for i, doc in enumerate(retrieved, 1):
        print(
            f"{i}. type={doc.metadata.get('type')}, "
            f"page={doc.metadata.get('page')}, "
            f"content={doc.page_content[:120]!r}"
        )

    print("\nPipeline answer:")
    answer = multimodal_pdf_rag_pipeline(query)
    print(answer)
    print("=" * 80)


Query: What does Figure 1 show about the Transformer architecture?
------------------------------------------------------------
Retrieval:
1. type=text, page=1, content='To the best of our knowledge, however, the Transformer is the first transduction model relying\nentirely on self-attentio'
2. type=text, page=1, content='In this work we propose the Transformer, a model architecture eschewing recurrence and instead\nrelying entirely on an at'
3. type=text, page=4, content='The Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries '
4. type=text, page=9, content='For translation tasks, the Transformer can be trained significantly faster than architectures based\non recurrent or conv'
5. type=text, page=2, content='Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-att'

Pipeline answer:
[{'type': 'text', 'text': 'Based on the provided text, Figure 1 illustra

### ❓ Preguntas finales
- Si tu PDF tiene **tablas**, ¿qué estrategia usarías: convertir a Markdown, embedding por fila, o OCR? ¿por qué?

R/= Depende de sí la tabla es una imagen o no. El OCR serviría para el caso de tablas en imágenes. Para el otro podría convertirse a Markdown para mejorar la lectura de la información de la tabla.


- ¿Cuándo Multi-Modal RAG NO vale la pena vs RAG textual?

R/= Cuando estemos trabajando con documentos que sean muy intensos en texto y tengan pocas imágenes (por ejemplo, ámbitos legales, donde casi todo es texto)
